In [1]:
class InsufficientFundsError(Exception):
    """Исключение: на счёте недостаточно средств."""

    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(
            f"недостаточно средств: баланс {balance:.2f}, "
            f"запрошено {amount:.2f}"
        )


class BankAccount:
    """Банковский счёт с историей операций."""

    BANK_PREFIX = "40817"
    count = 0
    all_accounts = []

    def __init__(self, owner, balance=0.0):
        """Создаёт счёт с владельцем и начальным балансом."""
        if not owner.strip():
            raise ValueError("владелец не может быть пустым")
        if balance < 0:
            raise ValueError("начальный баланс не может быть отрицательным")

        cls = type(self)
        cls.count += 1
        self.number = f"{cls.BANK_PREFIX}-{cls.count:04d}"
        self.owner = owner.strip()
        self.balance = float(balance)
        self.history = []
        cls.all_accounts.append(self)

    @classmethod
    def from_line(cls, line):
        """Создаёт счёт из строки вида «владелец;начальный баланс»."""
        parts = line.strip().split(";")
        if len(parts) != 2:
            raise ValueError(f"ожидалось 2 поля, получено {len(parts)}")
        owner, balance = parts
        return cls(owner.strip(), float(balance))

    @classmethod
    def apply_interest(cls, rate):
        """Начисляет процент rate на баланс всех созданных счетов."""
        if rate <= 0:
            raise ValueError("ставка должна быть положительной")
        for acc in cls.all_accounts:
            amount = round(acc.balance * rate / 100, 2)
            if amount > 0:
                acc._credit(amount, "проценты", f" ({rate}%)")

    def _credit(self, amount, op_type, note=""):
        """Внутреннее зачисление средств."""
        if amount <= 0:
            raise ValueError("сумма должна быть положительной")
        self.balance += amount
        self.history.append((op_type, amount, self.balance, note))

    def _debit(self, amount, op_type, note=""):
        """Внутреннее списание средств."""
        if amount <= 0:
            raise ValueError("сумма должна быть положительной")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        self.history.append((op_type, -amount, self.balance, note))

    def deposit(self, amount):
        """Пополняет счёт."""
        self._credit(amount, "пополнение")

    def withdraw(self, amount):
        """Снимает средства со счёта."""
        self._debit(amount, "снятие")

    def transfer(self, other, amount):
        """Переводит средства на другой счёт."""
        if not isinstance(other, BankAccount):
            raise TypeError("перевод возможен только на банковский счёт")
        self._debit(amount, "перевод", f" (на {other.number})")
        other._credit(amount, "перевод", f" (от {self.number})")

    def statement(self):
        """Возвращает текстовую выписку по счёту."""
        lines = [f"Счёт {self.number}, владелец {self.owner}"]
        if not self.history:
            lines.append("Операций нет")
        for i, (op, amount, balance, note) in enumerate(self.history, 1):
            lines.append(
                f"{i}. {op} {amount:+.2f} баланс {balance:.2f}{note}"
            )
        return "\n".join(lines)


def main():
    """Демонстрация работы класса BankAccount."""
    with open("accounts.txt", "w", encoding="utf-8") as f:
        f.write("Иванов И. И.;10000\n")
        f.write("Сидоров П. П.;800\n")
        f.write("Петрова А. С.;21000\n")
        f.write("Кузнецова М. В.;15000\n")
        f.write("Ошибочная строка;abc\n")

    accounts = []
    with open("accounts.txt", encoding="utf-8") as f:
        for no, line in enumerate(f, 1):
            try:
                accounts.append(BankAccount.from_line(line))
            except ValueError as e:
                print(f"Строка {no} пропущена: {e}")

    a1, a2, a3, a4 = accounts

    # 15 операций, включая ошибочные
    a1.deposit(5000)
    a1.withdraw(2000)
    a1.transfer(a3, 1500)

    try:
        a2.withdraw(1200)
    except InsufficientFundsError as e:
        print(f"Отказ: {a2.number} — {e}")

    try:
        a2.deposit(-100)
    except ValueError as e:
        print(f"Отказ: {a2.number} — {e}")

    a3.deposit(3000)
    a4.withdraw(5000)
    a2.deposit(700)

    try:
        a2.transfer(a4, 2000)
    except InsufficientFundsError as e:
        print(f"Отказ: {a2.number} — {e}")

    a4.deposit(2500)
    a1.withdraw(100)
    a3.transfer(a1, 500)

    try:
        a4.withdraw(20000)
    except InsufficientFundsError as e:
        print(f"Отказ: {a4.number} — {e}")

    a1.deposit(100)
    a2.withdraw(1500)

    for acc in accounts:
        print(acc.statement())
        print()

    total = sum(acc.balance for acc in accounts)
    best = max(accounts, key=lambda acc: acc.balance)
    print(
        f"Всего в банке: {total:.2f} | "
        f"наибольший остаток: {best.owner} ({best.balance:.2f})"
    )

    # Дополнительный пункт: начисление процентов на все счета
    BankAccount.apply_interest(5)
    total_after = sum(acc.balance for acc in accounts)
    print(f"После начисления 5%: всего в банке: {total_after:.2f}")


if __name__ == "__main__":
    main()

Строка 5 пропущена: could not convert string to float: 'abc'
Отказ: 40817-0002 — недостаточно средств: баланс 800.00, запрошено 1200.00
Отказ: 40817-0002 — сумма должна быть положительной
Отказ: 40817-0002 — недостаточно средств: баланс 1500.00, запрошено 2000.00
Отказ: 40817-0004 — недостаточно средств: баланс 12500.00, запрошено 20000.00
Счёт 40817-0001, владелец Иванов И. И.
1. пополнение +5000.00 баланс 15000.00
2. снятие -2000.00 баланс 13000.00
3. перевод -1500.00 баланс 11500.00 (на 40817-0003)
4. снятие -100.00 баланс 11400.00
5. перевод +500.00 баланс 11900.00 (от 40817-0003)
6. пополнение +100.00 баланс 12000.00

Счёт 40817-0002, владелец Сидоров П. П.
1. пополнение +700.00 баланс 1500.00
2. снятие -1500.00 баланс 0.00

Счёт 40817-0003, владелец Петрова А. С.
1. перевод +1500.00 баланс 22500.00 (от 40817-0001)
2. пополнение +3000.00 баланс 25500.00
3. перевод -500.00 баланс 25000.00 (на 40817-0001)

Счёт 40817-0004, владелец Кузнецова М. В.
1. снятие -5000.00 баланс 10000.00
